In [ ]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        """
        Initializes a Node in the Decision Tree.
        
        Args:
            feature (int, optional): The index of the feature to evaluate.
            threshold (float, optional): The threshold for the split.
            left (Node, optional): The left child Node (data <= threshold).
            right (Node, optional): The right child Node.
            value (int, optional): The predicted class label. Relevent only if this is a leaf.
        """
        # Attributes for Internal Nodes (Decision nodes)
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        
        # Attribute for Leaf Nodes (Terminal nodes)
        # We use a keyword-only argument (*) to prevent accidentally passing a value positionally
        self.value = value
        
    def is_leaf(self):
        """
        Helper method to determine if this node is a leaf.
        
        Returns:
            bool: True if it contains a prediction (value is not None), False otherwise.
        """
        return self.value is not None

In [ ]:
import numpy as np
from collections import Counter

class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=10, n_features=None, criterion='gini'):
        """
        Initializes the Decision Tree.
        
        Args:
            min_samples_split (int): Minimum number of samples required to split.
            max_depth (int): Maximum depth of the tree.
            n_features (int): Number of features to consider when looking for the best split.
            criterion (str): The function to measure the quality of a split. Supported criteria 
                             are 'gini' for Gini impurity and 'entropy' for information gain.
        """
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.n_features = n_features
        self.criterion = criterion
        self.root = None

    def fit(self, X, y):
        self.n_features = self.n_features if self.n_features else int(np.sqrt(X.shape[1]))
        self.root = self._grow_tree(X, y)

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))

        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        feat_idxs = np.random.choice(n_feats, self.n_features, replace=False)
        best_feature, best_thresh = self._best_split(X, y, feat_idxs)

        if best_feature is None:
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        left_idxs, right_idxs = self._split(X[:, best_feature], best_thresh)
        left_child = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right_child = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        
        return Node(feature=best_feature, threshold=best_thresh, left=left_child, right=right_child)

    def _best_split(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_threshold = None, None

        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)

            for thr in thresholds:
                # We now call a generalized gain function instead of just gini_gain
                gain = self._information_gain(y, X_column, thr)

                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_threshold = thr

        return split_idx, split_threshold

    def _information_gain(self, y, X_column, threshold):
        """Calculates the Gain of a proposed split based on the chosen criterion."""
        parent_impurity = self._calculate_impurity(y)

        left_idxs, right_idxs = self._split(X_column, threshold)
        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0
        
        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l = self._calculate_impurity(y[left_idxs])
        e_r = self._calculate_impurity(y[right_idxs])
        
        child_impurity = (n_l / n) * e_l + (n_r / n) * e_r
        gain = parent_impurity - child_impurity
        
        return gain

    def _calculate_impurity(self, y):
        """Acts as a switch board to use either Gini or Entropy."""
        if self.criterion == 'gini':
            return self._gini(y)
        elif self.criterion == 'entropy':
            return self._entropy(y)
        else:
            raise ValueError("Criterion must be 'gini' or 'entropy'")

    def _gini(self, y):
        counts = np.bincount(y)
        probabilities = counts / len(y)
        return 1.0 - np.sum(probabilities ** 2)

    def _entropy(self, y):
        counts = np.bincount(y)
        probabilities = counts / len(y)
        # Filter out 0 probabilities to prevent np.log2(0) from throwing a math domain error
        probabilities = probabilities[probabilities > 0]
        return -np.sum(probabilities * np.log2(probabilities))

    def _split(self, X_column, split_thresh):
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs

    def _most_common_label(self, y):
        counter = Counter(y)
        return counter.most_common(1)[0][0]

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value

        if x[node.feature] <= node.threshold: